# PyTorch BBBP Baseline

This notebook mirrors the first BBBP baseline in PyTorch using a simple character-level bag-of-symbols representation.

Task: binary classification on BBBP
Model: small feed-forward network trained with `BCEWithLogitsLoss`

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
bbbp = pd.read_csv(DATA_DIR / 'BBBP.csv')
display(bbbp.head())

In [ ]:
X = bbbp['smiles']
y = bbbp['p_np']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

In [ ]:
def build_vocab(smiles_series: pd.Series) -> dict[str, int]:
    symbols = sorted({char for smiles in smiles_series for char in smiles})
    return {symbol: index for index, symbol in enumerate(symbols)}


def encode_smiles(smiles_series: pd.Series, vocab: dict[str, int]) -> torch.Tensor:
    encoded = torch.zeros((len(smiles_series), len(vocab)), dtype=torch.float32)
    for row_index, smiles in enumerate(smiles_series):
        for char in smiles:
            if char in vocab:
                encoded[row_index, vocab[char]] += 1.0
        if len(smiles) > 0:
            encoded[row_index] /= float(len(smiles))
    return encoded


vocab = build_vocab(X_train)
X_train_tensor = encode_smiles(X_train.reset_index(drop=True), vocab)
X_valid_tensor = encode_smiles(X_valid.reset_index(drop=True), vocab)
X_test_tensor = encode_smiles(X_test.reset_index(drop=True), vocab)

y_train_tensor = torch.tensor(y_train.reset_index(drop=True).to_numpy(), dtype=torch.float32).unsqueeze(1)
y_valid_tensor = torch.tensor(y_valid.reset_index(drop=True).to_numpy(), dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.reset_index(drop=True).to_numpy(), dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid_tensor, y_valid_tensor), batch_size=128, shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Sequential(
    nn.Linear(len(vocab), 64),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(64, 1),
).to(device)

positive_fraction = float(y_train.mean())
pos_weight = torch.tensor([(1.0 - positive_fraction) / positive_fraction], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
history = []

for epoch in range(15):
    model.train()
    train_loss_total = 0.0
    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_total += float(loss.item()) * len(features)

    model.eval()
    valid_loss_total = 0.0
    valid_probs = []
    valid_targets = []
    with torch.no_grad():
        for features, labels in valid_loader:
            features = features.to(device)
            labels = labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            valid_loss_total += float(loss.item()) * len(features)
            valid_probs.extend(torch.sigmoid(logits).cpu().numpy().ravel().tolist())
            valid_targets.extend(labels.cpu().numpy().ravel().tolist())

    valid_pred = [1 if value >= 0.5 else 0 for value in valid_probs]
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss_total / len(X_train_tensor),
        'valid_loss': valid_loss_total / len(X_valid_tensor),
        'valid_accuracy': accuracy_score(valid_targets, valid_pred),
        'valid_f1': f1_score(valid_targets, valid_pred),
        'valid_roc_auc': roc_auc_score(valid_targets, valid_probs),
    })

history_df = pd.DataFrame(history)
display(history_df.tail().round(4))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
plt.plot(history_df['epoch'], history_df['valid_loss'], label='valid_loss')
plt.title('PyTorch BBBP Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor.to(device))
    test_probs = torch.sigmoid(test_logits).cpu().numpy().ravel()

test_pred = (test_probs >= 0.5).astype(int)
test_metrics = pd.DataFrame([
    {
        'accuracy': accuracy_score(y_test, test_pred),
        'f1': f1_score(y_test, test_pred),
        'roc_auc': roc_auc_score(y_test, test_probs),
    }
])
display(test_metrics.round(4))